gpg --list-secret-keys --keyid-format=long


In [ ]:
# pip install pypdf2 selenium webdriver-manager

In [ ]:
from PyPDF2 import PdfFileReader
import re

In [ ]:
pdf_path = "/Users/yaodingchen/Coding/HP407/Medline 1-100.pdf"
plain_path = "/Users/yaodingchen/Coding/HP407/records.txt"
with open(pdf_path, 'rb') as f:
    pdf = PdfFileReader(f)
    for page_no in range(pdf.numPages):
        page = pdf.getPage(page_no)
        text = page.extractText()
        with open(plain_path, "a") as writefile:
            writefile.write(text)

In [ ]:
URLs = []
key = '/Annots'
uri = '/URI'
ank = '/A'
with open(pdf_path, 'rb') as f:
    pdf = PdfFileReader(f)
    for page_no in range(pdf.numPages):
        page = pdf.getPage(page_no)
        objs = page.getObject()
        if key in objs.keys():
            ann = objs[key]
            for a in ann:
                u = a.getObject()
                if uri in u[ank].keys():
                    url = u[ank][uri]
                    if "exlibrisgroup" in url:
                        URLs.append(url)

In [ ]:
len(URLs)

In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
import time
from tqdm import tqdm

#browser=webdriver.Chrome(service=Service(ChromeDriverManager().install()))
chrome_options=webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
desired_capabilities = DesiredCapabilities.CHROME
desired_capabilities["pageLoadStrategy"] = "none"
#browser=webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
browser=webdriver.Chrome(service=Service(ChromeDriverManager().install()))

for url in tqdm(URLs):
    
    browser.get(url)
    
    timeout=20
    #element_present = EC.presence_of_element_located((By.ID, 'RISPushToButtonFullView'))
    element_present = EC.presence_of_element_located((By.ID, 'BibTeXPushToButtonFullView'))
    WebDriverWait(browser, timeout).until(element_present)
    #ris_button = browser.find_element("id","RISPushToButtonFullView")
    #ris_button.click()
    bib_button = browser.find_element("id","BibTeXPushToButtonFullView")
    bib_button.click()

    element_present = EC.presence_of_element_located((By.XPATH, "/html/body/primo-explore/div/prm-services-page/prm-full-view-cont/md-content/div[2]/prm-full-view/div/div/div/div[1]/div/div[3]/div/prm-full-view-service-container/div[2]/prm-action-list/prm-action-container/prm-export-bibtex/div/md-content/form/div[2]/div/button/span"))
    #dowload_button = browser.find_element("xpath", "/html/body/primo-explore/div/prm-services-page/prm-full-view-cont/md-content/div[2]/prm-full-view/div/div/div/div[1]/div/div[3]/div/prm-full-view-service-container/div[2]/prm-action-list/prm-action-container/prm-export-ris/div/md-content/form/div[2]/div/button")
    dowload_button = browser.find_element("xpath", "/html/body/primo-explore/div/prm-services-page/prm-full-view-cont/md-content/div[2]/prm-full-view/div/div/div/div[1]/div/div[3]/div/prm-full-view-service-container/div[2]/prm-action-list/prm-action-container/prm-export-bibtex/div/md-content/form/div[2]/div/button/span")
    dowload_button.click()
    time.sleep(.5)
    
browser.quit()